[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C00_VLM_Multimodal_Course/07_evaluation/07_evaluation.ipynb)

# 07 · VLM 评测体系 (Evaluating Vision-Language Models)

本 notebook 从零搭一套**最小但方法论完整**的 VLM 评测 harness，配套讲解见 `07_讲解.html`。我们用一个小 VLM（`Qwen/Qwen2-VL-2B-Instruct`，fp16）跑通以下评测组件：

1. **多选 VQA 评测集**：手工构造 6 条 + 可选从 HuggingFace 加载真实基准（`Lin-Chen/MMStar`）切片。
2. **答案解析**：正则抽取选项字母 + 兜底链，区分 `unparseable`（指令遵循失败）与真错。
3. **CircularEval**：选项循环移位多轮、全对才算对，量化 **position bias**（位置偏差）。
4. **污染 / 语言先验探针**：去掉图像再问一遍，统计"不看图也能答对"的比例（MMStar 的 MG/ML 思想）。
5. **LLM-as-a-Judge**：对开放式 caption 任务打软分；检测到 `OPENAI_API_KEY` 则调真 API，否则走本地 rubric 占位并说明局限。
6. **结果汇总表**：naive acc / circular acc / no-image acc 对比 + 小结 + 练习。
7. **4 道 ✏️ 练习**：extract_choice / CircularEval / position bias / judge 打分解析——纯 Python 逻辑，无需模型与网络即可完成并自测。

> **算力**：Qwen2-VL-2B fp16 约需 **5–6 GB 显存**，Colab T4（16GB）够用，默认不量化；纯 CPU/MPS 也能跑但很慢。无 GPU 时可只读代码、把推理 cell 跳过看后面的方法论。
> **不需要联网下载模型也能学方法论**：解析、CircularEval、探针、judge 这些逻辑函数本身可在无模型环境下用伪造输出做单元测试（见对应 cell）。

In [ ]:
# === Cell 1: import + device + 版本 ===
import os, re, json, math, random
import torch

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"使用设备 device = {device}")
print(f"torch version = {torch.__version__}")

try:
    import transformers
    print(f"transformers version = {transformers.__version__}")
except Exception as e:
    print("transformers 未安装：", e)

# 2B 模型 fp16 只要 ~5-6GB，T4(16GB) 默认不需要量化；显存特别紧张时可自己改成 True
USE_4BIT = False
print(f"USE_4BIT (bitsandbytes 4-bit) = {USE_4BIT}")

random.seed(0)
torch.manual_seed(0)

## 1. 定义一个小型多选 VQA 评测集

每条题包含：图像 URL、问题、4 个选项 A/B/C/D、正确答案字母。这里手工构造 6 条覆盖**计数 / OCR / 颜色 / 空间 / 常识 / 属性**等不同视觉能力维度——多维能力是 VLM 评测的核心（见讲解 §1）。

我们把每题表示成一个 `dict`，并写一个轻量数据类便于后续做选项循环移位（CircularEval）。

In [ ]:
# === Cell 2: 手工构造多选评测集 ===
# 图像来自 COCO val2017 公共 URL（稳定可访问）。答案字母对应 options 列表的下标 A=0,B=1,C=2,D=3。
MANUAL_ITEMS = [
    {
        "image": "http://images.cocodataset.org/val2017/000000039769.jpg",  # 两只猫躺在沙发上，旁边有遥控器
        "question": "How many cats are lying on the couch in the image?",
        "options": ["One", "Two", "Three", "Four"],
        "answer": "B",
        "skill": "counting",
    },
    {
        "image": "http://images.cocodataset.org/val2017/000000039769.jpg",
        "question": "What objects are placed next to the cats on the couch?",
        "options": ["Books", "Remote controls", "Cups", "Phones"],
        "answer": "B",
        "skill": "recognition",
    },
    {
        "image": "http://images.cocodataset.org/val2017/000000000285.jpg",  # 一只棕熊
        "question": "What animal is shown in the image?",
        "options": ["A dog", "A bear", "A horse", "A cow"],
        "answer": "B",
        "skill": "recognition",
    },
    {
        "image": "http://images.cocodataset.org/val2017/000000000632.jpg",  # 室内场景
        "question": "Is the scene in the image indoors or outdoors?",
        "options": ["Indoors", "Outdoors", "Underwater", "In space"],
        "answer": "A",
        "skill": "scene",
    },
    {
        "image": "http://images.cocodataset.org/val2017/000000252219.jpg",  # 比萨斜塔/街景类
        "question": "What is the dominant type of object in this street scene?",
        "options": ["Boats", "Buildings", "Airplanes", "Farm animals"],
        "answer": "B",
        "skill": "scene",
    },
    {
        "image": "http://images.cocodataset.org/val2017/000000037777.jpg",  # 餐桌食物
        "question": "What category best describes the items on the table?",
        "options": ["Food", "Electronics", "Clothing", "Tools"],
        "answer": "A",
        "skill": "recognition",
    },
]

LETTERS = ["A", "B", "C", "D"]
print(f"手工评测集大小 = {len(MANUAL_ITEMS)}")
print("第一题示例：")
print(json.dumps(MANUAL_ITEMS[0], ensure_ascii=False, indent=2))

## 1b. （可选）从 HuggingFace 加载真实基准切片

下面尝试从 `datasets` 加载 **MMStar**（`Lin-Chen/MMStar`，[Chen 2024]）的前几条作为真实样本，并适配成我们的统一格式。MMStar 是专为"必须看图"净化过的多选基准，正适合本 notebook 的污染探针。

**用 try/except 回退**：若无网络 / 数据集结构有变 / 字段不匹配，则继续用上面的手工集，不影响后续。MMStar 的字段一般为 `image`（PIL）、`question`（题干已内嵌选项）、`answer`（字母）。

In [ ]:
# === Cell 3: 尝试加载真实基准切片，失败则回退手工集 ===
EVAL_ITEMS = list(MANUAL_ITEMS)   # 默认用手工集
DATA_SOURCE = "manual"

def parse_options_from_question(q):
    # MMStar 把选项写在题干里，形如 "...\nA. xxx\nB. yyy\nC. ...\nD. ..."
    opts = re.findall(r"(?m)^\s*([A-D])[\.\)]\s*(.+?)\s*$", q)
    if len(opts) >= 2:
        body = re.split(r"(?m)^\s*[A-D][\.\)]", q)[0].strip()
        return body, [o[1] for o in opts]
    return q, None

try:
    from datasets import load_dataset
    ds = load_dataset("Lin-Chen/MMStar", split="val", streaming=True)
    loaded = []
    for ex in ds:
        body, opts = parse_options_from_question(ex.get("question", ""))
        ans = str(ex.get("answer", "")).strip().upper()
        if opts and len(opts) >= 2 and ans in LETTERS[:len(opts)]:
            loaded.append({
                "image": ex["image"],          # PIL.Image，下游加载需兼容
                "question": body,
                "options": (opts + ["", "", ""])[:4] if len(opts) < 4 else opts[:4],
                "answer": ans,
                "skill": ex.get("category", "mmstar"),
            })
        if len(loaded) >= 5:
            break
    if loaded:
        EVAL_ITEMS = loaded
        DATA_SOURCE = "MMStar(Lin-Chen/MMStar)"
        print(f"成功加载真实基准 {DATA_SOURCE}，取 {len(EVAL_ITEMS)} 条")
    else:
        print("MMStar 加载但字段未匹配，回退手工集")
except Exception as e:
    print(f"加载真实基准失败，回退手工集。原因：{type(e).__name__}: {e}")

print(f"\n最终数据来源 DATA_SOURCE = {DATA_SOURCE}，样本数 = {len(EVAL_ITEMS)}")

## 2. 加载小 VLM：`Qwen/Qwen2-VL-2B-Instruct`（fp16）

用 `Qwen2VLForConditionalGeneration` + `AutoProcessor` 加载，GPU 上默认 fp16。Qwen2-VL 用 chat-template 拼接图文消息，是当前主流的多模态对话接口。

> 标注：fp16 参数约 **~5GB**，加上激活略高一些，Colab T4（16GB）单卡够用，不需要量化。`USE_4BIT` 只是留给显存特别紧张场景的开关，默认关闭。无 GPU 时模型加载会很慢，可只读不跑。

In [ ]:
# === Cell 4: 加载模型与 processor ===
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
model, processor = None, None

try:
    from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

    quant_kwargs = {}
    if USE_4BIT:
        from transformers import BitsAndBytesConfig
        quant_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if device != "cpu" else torch.float32,
        device_map="auto" if device == "cuda" else None,
        **quant_kwargs,
    )
    if device != "cuda":
        model = model.to(device)
    model.eval()

    # min/max pixels 控制视觉 token 数，省显存（见模块 06 任意分辨率）
    processor = AutoProcessor.from_pretrained(
        MODEL_ID, min_pixels=256*28*28, max_pixels=768*28*28
    )
    print(f"模型加载成功：{MODEL_ID}")
except Exception as e:
    print(f"模型加载失败（无 GPU/无网络时正常）：{type(e).__name__}: {e}")
    print("→ 后续推理 cell 会被跳过；解析/CircularEval/探针/judge 的逻辑函数仍可用伪造输出测试。")

## 3. 按多选格式构造 prompt 并生成回答

关键点（见讲解 §3.3）：prompt 要给**极明确的格式约束**——"只回答选项字母"——并尽量减少指令遵循失败。我们写两个函数：

- `build_mc_prompt(item, options)`：把题干 + 选项渲染成多选 prompt（`options` 可被打乱，供 CircularEval 复用）。
- `vlm_answer(item, options, with_image=True)`：调用模型生成；`with_image=False` 时**去掉图像**（污染/语言先验探针用）。

图像加载兼容两种情况：URL 字符串（手工集）或已是 PIL.Image（MMStar）。

In [ ]:
# === Cell 5: prompt 构造 + 图像加载 + 生成 ===
from PIL import Image
import requests
from io import BytesIO

def load_image(img):
    if isinstance(img, Image.Image):
        return img.convert("RGB")
    if isinstance(img, str):
        if img.startswith("http"):
            return Image.open(BytesIO(requests.get(img, timeout=30).content)).convert("RGB")
        return Image.open(img).convert("RGB")
    raise ValueError(f"无法识别的图像类型: {type(img)}")

INSTRUCTION = ("Answer with the single letter of the correct option only "
               "(A, B, C, or D). Do not explain.")

def build_mc_prompt(item, options):
    lines = [item["question"], ""]
    for letter, opt in zip(LETTERS, options):
        lines.append(f"{letter}. {opt}")
    lines.append("")
    lines.append(INSTRUCTION)
    return "\n".join(lines)

@torch.no_grad()
def vlm_answer(item, options, with_image=True, max_new_tokens=16):
    if model is None or processor is None:
        raise RuntimeError("模型未加载，无法生成。")
    prompt_text = build_mc_prompt(item, options)

    if with_image:
        content = [{"type": "image"}, {"type": "text", "text": prompt_text}]
        images = [load_image(item["image"])]
    else:
        # 去图探针：完全不放 image，仅文本（语言先验基线）
        content = [{"type": "text", "text": prompt_text}]
        images = None

    messages = [{"role": "user", "content": content}]
    chat_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[chat_text], images=images, return_tensors="pt").to(model.device)

    gen = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)  # greedy: 可复现
    trimmed = gen[:, inputs["input_ids"].shape[1]:]
    out = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
    return out

# 冒烟测试：模型可用时跑第一题
if model is not None:
    raw = vlm_answer(EVAL_ITEMS[0], EVAL_ITEMS[0]["options"], with_image=True)
    print("第一题原始输出：", repr(raw))
else:
    print("模型未加载，跳过冒烟测试。")

## 4. 答案解析：正则抽取 + 兜底链

实现讲解 §3.2 的**降级匹配链**：先严后宽，找不到则标 `unparseable`（绝不静默判错）。返回抽取到的字母或 `None`。

`unparseable` 比例本身是一个独立诊断维度——高 `unparseable` 往往是**指令遵循失败**而非能力失败。

In [ ]:
# === Cell 6: 答案抽取函数（带兜底链）===
def extract_choice(text, options=None):
    if text is None:
        return None
    t = text.strip()

    # ① 严格：开头就是 A / A. / (A) / A)
    m = re.match(r"^\s*[\(\[]?([A-D])[\)\].:,]?\s*", t)
    if m:
        return m.group(1).upper()

    # ② 模式："answer is B" / "答案是 B" / "选 C" / "option D"
    m = re.search(r"(?:answer|option|choice|答案|选项|选)\s*(?:is|：|:)?\s*[\(\[]?([A-D])\b",
                  t, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()

    # ③ 宽松：全文只出现唯一一个孤立选项字母
    isolated = re.findall(r"(?<![A-Za-z])([A-D])(?![A-Za-z])", t)
    uniq = set(isolated)
    if len(uniq) == 1:
        return isolated[0].upper()

    # ④ 选项文本匹配：回答里完整包含某个选项文本
    if options:
        hits = [LETTERS[i] for i, opt in enumerate(options)
                if opt and opt.lower() in t.lower()]
        if len(set(hits)) == 1:
            return hits[0]

    # ⑤ 兜底：无法解析
    return None

# 单元测试：用伪造输出验证抽取链（无需模型）
_tests = [
    ("B", "B"),
    ("(C).", "C"),
    ("The answer is D because the bear is brown.", "D"),
    ("答案是 A", "A"),
    ("I think it could be B or maybe C", None),   # 两个候选 → 矛盾 → unparseable
    ("It is clearly a bear.", None),              # 无字母（选项文本匹配在调用处生效）
]
print("抽取链单元测试：")
for inp, exp in _tests:
    got = extract_choice(inp)
    flag = "OK " if got == exp else "XX "
    print(f"  {flag} extract({inp!r:55}) = {got!r:6} (期望 {exp!r})")

## 5. 朴素 accuracy（naive accuracy）

对每题用**原始选项顺序**问一次、抽取、与金标准比对。同时统计 `unparseable` 比例。模型不可用时，用一个**可控的伪模型**（mock）模拟"带位置偏差 + 偶尔不遵循格式"的输出，让后续所有指标都能在无 GPU 环境下跑出数字、看清逻辑。

In [ ]:
# === Cell 7: 朴素 accuracy + mock 回退 ===
# mock 模型：模拟一个"偏好选 B、且 20% 概率不按格式"的弱 VLM。
# 它不真的看图——这正好让 no-image 探针(§7)能演示"语言先验"现象。
_mock_rng = random.Random(42)
def mock_answer(item, options, with_image=True):
    # 偏好把答案押在位置 B（典型 position bias）
    biased = "B"
    if _mock_rng.random() < 0.20:                 # 20% 指令不遵循
        return f"I believe the correct choice is {biased}, since it fits best."
    if _mock_rng.random() < 0.15:                 # 15% 直接 unparseable
        return "It is hard to tell from the picture."
    return biased

def get_answer(item, options, with_image=True):
    if model is not None:
        return vlm_answer(item, options, with_image=with_image)
    return mock_answer(item, options, with_image=with_image)

def run_naive(items):
    records = []
    for it in items:
        raw = get_answer(it, it["options"], with_image=True)
        pred = extract_choice(raw, it["options"])
        records.append({
            "question": it["question"][:50],
            "gold": it["answer"],
            "raw": raw,
            "pred": pred,
            "correct": (pred == it["answer"]),
            "unparseable": (pred is None),
        })
    n = len(records)
    acc = sum(r["correct"] for r in records) / n
    unparse = sum(r["unparseable"] for r in records) / n
    return records, acc, unparse

naive_records, naive_acc, naive_unparse = run_naive(EVAL_ITEMS)
print(f"模型来源: {'真实 VLM' if model is not None else 'MOCK(伪模型)'}")
print(f"朴素 accuracy = {naive_acc:.3f}")
print(f"unparseable 比例 = {naive_unparse:.3f}  (高 → 可能是指令遵循失败而非能力失败)")
print("\n逐题：")
for r in naive_records:
    print(f"  gold={r['gold']} pred={str(r['pred']):4} correct={r['correct']!s:5} raw={r['raw']!r}")

## 6. CircularEval：循环移位多轮、全对才算对

实现讲解 §4 的公式。对每题做 $k$ 次**循环移位（cyclic shift）**：第 $j$ 次把选项整体右移 $j$ 位，正确答案的位置随之变到 $\sigma_j(c^\star)$。**$k$ 次全部答对**才算这题对（连乘指示函数）。

对比 `circular acc` 与 `naive acc`：差值越大 → 模型越依赖 position bias（位置偏差），视觉/推理越脆弱。

In [ ]:
# === Cell 8: CircularEval ===
def cyclic_shift(options, gold_letter, j):
    # 把 options 整体右移 j 位；返回 (新options, 新正确字母)
    k = len(options)
    shifted = [options[(i - j) % k] for i in range(k)]   # 新位置 i 放原来的 (i-j)
    gold_idx = LETTERS.index(gold_letter)
    new_gold_idx = (gold_idx + j) % k
    return shifted, LETTERS[new_gold_idx]

def run_circular(items):
    records = []
    for it in items:
        k = len(it["options"])
        passes = []
        all_correct = True
        for j in range(k):
            opts_j, gold_j = cyclic_shift(it["options"], it["answer"], j)
            raw = get_answer(it, opts_j, with_image=True)
            pred = extract_choice(raw, opts_j)
            ok = (pred == gold_j)
            passes.append((j, gold_j, pred, ok))
            all_correct = all_correct and ok
        records.append({"question": it["question"][:50],
                        "circular_correct": all_correct, "passes": passes})
    acc = sum(r["circular_correct"] for r in records) / len(records)
    return records, acc

circ_records, circ_acc = run_circular(EVAL_ITEMS)
print(f"CircularEval accuracy = {circ_acc:.3f}")
print(f"朴素 accuracy        = {naive_acc:.3f}")
print(f"差值 (naive - circular) = {naive_acc - circ_acc:.3f}  ← 越大说明位置偏差越严重")
print("\n逐题各 pass（位置一变就答错 = 位置偏差暴露）：")
for r in circ_records:
    seq = "  ".join(f"pass{j}:gold={g},pred={p},{'✓' if ok else '✗'}" for j,g,p,ok in r["passes"])
    print(f"  [{('✓' if r['circular_correct'] else '✗')}] {r['question']}\n      {seq}")

## 7. 污染 / 语言先验探针：去掉图像再问一遍

实现讲解 §6 的 **answer-without-image 基线**。同一批题，把图像抽掉、只喂文本，统计还能答对多少：

- `acc_wo`（without-vision）越接近随机基线 → benchmark 越"干净"。
- `acc_wo` 远高于随机 → 题目靠语言先验即可答 / 可能数据泄漏。

并算 MMStar 的两个净化指标：**MG = acc_wv − acc_wo**（视觉真实增益，越大越好）、**ML = max(0, acc_wo − acc_rand)**（多模态泄漏，越小越好）。随机基线 = 1/选项数。

In [ ]:
# === Cell 9: no-image 探针 + MG/ML ===
def run_no_image(items):
    records = []
    for it in items:
        raw = get_answer(it, it["options"], with_image=False)
        pred = extract_choice(raw, it["options"])
        records.append({"gold": it["answer"], "pred": pred,
                        "correct": (pred == it["answer"]), "raw": raw})
    acc = sum(r["correct"] for r in records) / len(records)
    return records, acc

noimg_records, acc_wo = run_no_image(EVAL_ITEMS)
acc_wv = naive_acc                                   # with-vision = 朴素 acc
acc_rand = 1.0 / len(EVAL_ITEMS[0]["options"])       # 4 选项 → 0.25

MG = acc_wv - acc_wo
ML = max(0.0, acc_wo - acc_rand)

print(f"acc_with_vision (acc_wv)    = {acc_wv:.3f}")
print(f"acc_without_image (acc_wo)  = {acc_wo:.3f}")
print(f"random baseline (acc_rand)  = {acc_rand:.3f}")
print(f"\nMG (multi-modal gain,  越大越好) = {MG:.3f}   ← 看图带来的真实增益")
print(f"ML (multi-modal leakage, 越小越好) = {ML:.3f}   ← 不看图就能答的'水分'")
if ML > 0.15:
    print("⚠️ ML 偏高：该批题在此模型上存在明显语言先验/泄漏，分数不全是视觉能力。")
else:
    print("✓ ML 较低：去图后接近随机，视觉成分较干净。")

## 8. LLM-as-a-Judge：对开放式 caption 任务打分

开放生成无法 exact-match（讲解 §5）。我们对一个 caption 任务用 judge 打 0–5 软分：

- 若检测到 `OPENAI_API_KEY` → 调 `openai` 客户端，用 **reasoning-before-score 的 rubric prompt**（真实 LLM judge）。
- 否则 → 走**本地 rubric 占位**：按"参考答案关键词命中率 + 长度合理性"给分，并**明确说明其局限**（无法判断语义正确性、易被关键词堆砌欺骗、不能替代真 judge）。

下面先打印完整 judge prompt，再实现两条路径。

In [ ]:
# === Cell 10: judge prompt 全文 + 评分函数 ===
JUDGE_SYSTEM = "你是严格的视觉描述评分员。只依据参考答案判定，不臆测，忽略表述风格与长度。"

JUDGE_PROMPT_TEMPLATE = '''请给下面的图像描述打分（0-5 整数）。

评分量纲（rubric）:
  5 = 完全正确且覆盖参考答案的关键要素
  4 = 正确，遗漏个别次要要素
  3 = 部分正确，遗漏或弄错某个关键要素
  2 = 大部分错误，仅个别正确
  1 = 基本错误
  0 = 答非所问 / 拒答 / 与图无关

Question: {question}
Reference answer: {reference}
Model answer: {prediction}

要求：先写一句 reasoning 说明依据，再在**单独一行**输出  Score: <0-5>。
评分时忽略长度与文采，只看与参考答案的语义一致性。'''

print("===== JUDGE PROMPT 全文 =====")
print("[System]", JUDGE_SYSTEM)
print(JUDGE_PROMPT_TEMPLATE)

In [ ]:
# === Cell 11: 两条评分路径（真实 API / 本地占位）===
def parse_score(text):
    m = re.search(r"Score:\s*([0-5])", text)
    if m:
        return int(m.group(1))
    m = re.search(r"\b([0-5])\b", text)   # 兜底：抓第一个 0-5
    return int(m.group(1)) if m else None

def judge_openai(question, reference, prediction, model_name="gpt-4o-mini"):
    from openai import OpenAI
    client = OpenAI()
    prompt = JUDGE_PROMPT_TEMPLATE.format(question=question, reference=reference, prediction=prediction)
    resp = client.chat.completions.create(
        model=model_name, temperature=0,
        messages=[{"role": "system", "content": JUDGE_SYSTEM},
                  {"role": "user", "content": prompt}],
    )
    text = resp.choices[0].message.content
    return parse_score(text), text

def judge_local(question, reference, prediction):
    '''本地 rubric 占位：关键词命中率 + 长度合理性。
    局限（必须声明）：①只看字面关键词，无法判断语义对错；②可被关键词堆砌欺骗；
    ③对同义改写零分宽容；④绝不能替代真 LLM judge 或人工，仅用于无 API 时跑通流程。'''
    ref_words = set(re.findall(r"[a-zA-Z一-鿿]+", reference.lower()))
    pred_words = set(re.findall(r"[a-zA-Z一-鿿]+", prediction.lower()))
    if not ref_words:
        return 0, "no reference keywords"
    hit = len(ref_words & pred_words) / len(ref_words)
    length_ok = 3 <= len(pred_words) <= 60
    score = round(5 * hit) if length_ok else max(0, round(5 * hit) - 1)
    rationale = f"[LOCAL PLACEHOLDER] keyword hit={hit:.2f}, length_ok={length_ok} → {score}"
    return int(score), rationale

USE_OPENAI_JUDGE = bool(os.environ.get("OPENAI_API_KEY"))
def judge(question, reference, prediction):
    if USE_OPENAI_JUDGE:
        try:
            return judge_openai(question, reference, prediction)
        except Exception as e:
            print(f"OpenAI judge 失败，回退本地占位：{e}")
    return judge_local(question, reference, prediction)

print(f"judge 路径 = {'OpenAI API (真实 LLM judge)' if USE_OPENAI_JUDGE else '本地 rubric 占位（见局限说明）'}")

: 

In [ ]:
# === Cell 12: 在一个 caption 任务上跑 judge ===
CAPTION_TASK = {
    "image": "http://images.cocodataset.org/val2017/000000039769.jpg",
    "question": "Describe the image in one sentence.",
    "reference": "Two cats are lying on a pink couch with two remote controls next to them.",
}

# 取模型 caption；模型不可用时用一组伪造候选演示 judge 的区分度
if model is not None:
    cap_item = {**CAPTION_TASK, "options": []}
    messages = [{"role": "user", "content": [{"type": "image"},
                 {"type": "text", "text": CAPTION_TASK["question"]}]}]
    chat_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[chat_text], images=[load_image(CAPTION_TASK["image"])],
                       return_tensors="pt").to(model.device)
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    pred_caption = processor.batch_decode(gen[:, inputs["input_ids"].shape[1]:],
                                          skip_special_tokens=True)[0].strip()
    candidates = [pred_caption]
else:
    candidates = [
        "Two cats lie on a pink couch next to two remote controls.",   # 好
        "A couple of cats are resting on a sofa.",                      # 部分（漏遥控器）
        "A dog is running in a green park under the sun.",              # 错
    ]

print("===== JUDGE 评分结果 =====")
for cap in candidates:
    s, rationale = judge(CAPTION_TASK["question"], CAPTION_TASK["reference"], cap)
    print(f"\nprediction: {cap}")
    print(f"score = {s}")
    print(f"rationale: {rationale}")

## 9. 汇总结果表 + 小结

把三类指标拢到一张表：**naive acc / circular acc / no-image acc**，外加 unparseable 比例与 MG/ML。这正是一份最小评测报告该有的样子——配上置信区间（这里样本太小，仅演示点估计；真实评测请按讲解 §7 用 bootstrap 给 CI）。

In [ ]:
# === Cell 13: 汇总表 ===
try:
    import pandas as pd
    summary = pd.DataFrame([{
        "data_source": DATA_SOURCE,
        "n": len(EVAL_ITEMS),
        "naive_acc": round(naive_acc, 3),
        "circular_acc": round(circ_acc, 3),
        "no_image_acc": round(acc_wo, 3),
        "random_baseline": round(acc_rand, 3),
        "unparseable_rate": round(naive_unparse, 3),
        "MG(gain)": round(MG, 3),
        "ML(leakage)": round(ML, 3),
    }])
    print(summary.to_string(index=False))
except Exception:
    print(f"data_source={DATA_SOURCE} n={len(EVAL_ITEMS)} "
          f"naive={naive_acc:.3f} circular={circ_acc:.3f} no_image={acc_wo:.3f} "
          f"rand={acc_rand:.3f} unparse={naive_unparse:.3f} MG={MG:.3f} ML={ML:.3f}")

print("\n如何读这张表：")
print("• naive > circular 的差值 = 位置偏差(position bias)带来的水分")
print("• no_image 接近 random = benchmark 干净；远高于 random = 语言先验/泄漏(ML 高)")
print("• MG 高且 ML 低 = 这批题真的在测视觉")
print("• unparseable 高 = 指令遵循失败，需把这些样本捞出来人工/LLM 复核，别当能力失败")

---
## ✏️ 练习 1：重写鲁棒的 `extract_choice_ex`

不翻上文，自己重写一遍答案抽取链：输入模型的自由文本回答 `text`（可选传入 `options` 列表），返回抽到的选项字母 `"A"`–`"D"`，抽不出返回 `None`（unparseable，绝不静默判错）。按**先严后宽**的降级链：

1. **开头即字母**：`"B"`、`"(C)."`、`"A) ..."`——注意字母后**不能紧跟其他字母**，否则 `"Because..."` 会被误抽成 B。
2. **模式词**：`"The answer is (B)"`、`"答案是 A"`、`"option D"`（关键词 answer/option/choice/答案/选项/选）。
3. **全文唯一孤立字母**：整段里只出现一个孤立的 A–D 才返回；出现两个不同字母（`"B or maybe C"`）→ 矛盾 → 继续降级。
4. **选项文本匹配**：回答里唯一命中某条选项文本（不区分大小写）→ 返回对应字母。
5. 全部失败 → `None`。

**提示**：四级分别用 `re.match` / `re.search` / `re.findall` / 列表推导；孤立字母用 `(?<![A-Za-z])([A-D])(?![A-Za-z])`；①的"不紧跟字母"用 `(?![A-Za-z])`。约 25 行。边界：`text=None` 直接返回 `None`。

In [ ]:
import re

def extract_choice_ex(text, options=None):
    # TODO: ① re.match 开头字母，允许 (A) [A] A. A) 等包装，字母后不能紧跟字母
    # TODO: ② re.search 模式词 answer/option/choice/答案/选项/选 后跟字母
    # TODO: ③ re.findall 孤立字母；全文唯一才返回，两个不同字母 → 矛盾，继续降级
    # TODO: ④ options 文本子串匹配（不区分大小写）；唯一命中才返回对应字母
    # TODO: ⑤ 全部失败返回 None
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert extract_choice_ex("B") == "B"
assert extract_choice_ex("(C).") == "C"
assert extract_choice_ex("The answer is (B), because the cat is black.") == "B"
assert extract_choice_ex("答案是 A") == "A"
assert extract_choice_ex("I think it could be B or maybe C") is None        # 矛盾 → unparseable
assert extract_choice_ex("It is a bear.", options=["cat", "dog", "bear", "fish"]) == "C"
assert extract_choice_ex("Because the sky is blue, I cannot decide.") is None  # "Because" 不是 B
assert extract_choice_ex(None) is None                                       # 边界：空输入
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 CircularEval（轮转 + 全对才算对）

CircularEval 就两件事，把它们都写出来：

- `cyclic_shift_ex(options, gold_letter, j)`：把 `options` 整体**循环右移** `j` 位（新位置 $i$ 放原位置 $(i-j) \bmod k$ 的选项），正确答案下标随之 $+j \bmod k$，返回 `(新options, 新正确字母)`。
- `run_circular_ex(items, answer_fn)`：每条 item 形如 `{"options": [...], "answer": "C"}`；对每题做 $k$ 次轮转（$j=0..k-1$），每次调用 `answer_fn(item, opts_j)` 得到预测字母，与轮转后的金标比对；**$k$ 次全部答对**这题才记对，返回 accuracy（float）。

**提示**：字母表用 `"ABCD"`；不变量——任意 `j` 下 `新options["ABCD".index(新字母)]` 必须仍是原正确选项文本。两个函数合计约 15 行。

In [ ]:
def cyclic_shift_ex(options, gold_letter, j):
    # TODO: 新位置 i 放原位置 (i - j) % k 的选项；gold 下标 (idx + j) % k → 新字母
    raise NotImplementedError

def run_circular_ex(items, answer_fn):
    # TODO: 对每题 j = 0..k-1 轮转，pred = answer_fn(item, opts_j)
    #       与轮转后的 gold 比对；k 次全部正确才算这题对；返回 accuracy
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
opts = ["cat", "dog", "bear", "fish"]
assert cyclic_shift_ex(opts, "C", 0) == (["cat", "dog", "bear", "fish"], "C")
assert cyclic_shift_ex(opts, "C", 1) == (["fish", "cat", "dog", "bear"], "D")
for j in range(4):                                   # 不变量：正确字母始终指向 "bear"
    o2, g2 = cyclic_shift_ex(opts, "C", j)
    assert o2["ABCD".index(g2)] == "bear"

items = [
    {"options": ["cat", "dog", "bear", "fish"], "answer": "C"},
    {"options": ["one", "two", "three", "four"], "answer": "A"},
]
def oracle(item, options):                            # 真懂题：按选项文本找答案，不怕轮转
    gold_text = item["options"]["ABCD".index(item["answer"])]
    return "ABCD"[options.index(gold_text)]
always_b = lambda item, options: "B"                  # 只会押位置 B 的"作弊"模型
def mixed(item, options):                             # 只真懂第一题
    return oracle(item, options) if "cat" in item["options"] else "B"

assert run_circular_ex(items, oracle) == 1.0          # 真能力不怕轮转
assert run_circular_ex(items, always_b) == 0.0        # 位置偏差被 CircularEval 清零
assert run_circular_ex(items, mixed) == 0.5
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 position bias 度量 `position_bias_ex`

光知道 circular acc 低还不够，要能**定位**偏差：实现 `position_bias_ex(preds)`，输入一串预测字母（可含 `None` 表示 unparseable），返回 `(rates, bias)`：

- `rates`：dict，`"A"`–`"D"` 每个位置的**被选率**——分母是**可解析**的预测数（`None` 不进分母）；
- `bias`：`max(rates.values()) - 0.25`，即最热位置相对均匀基线（$1/4$）的超出量。无偏模型 ≈ 0，"全押 B" = 0.75。

**提示**：先过滤 `None`，再对 `"ABCD"` 各算 `count/总数`；边界——全是 `None`（或空列表）时返回 `({"A": 0.0, "B": 0.0, "C": 0.0, "D": 0.0}, 0.0)`。10 行以内。

In [ ]:
def position_bias_ex(preds):
    # TODO: 过滤 None；对 "ABCD" 各位置算被选率（分母 = 可解析预测数）
    # TODO: bias = max(rates) - 0.25；全 None / 空输入 → 全 0.0
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rates, bias = position_bias_ex(["B", "B", "B", "B"])
assert rates == {"A": 0.0, "B": 1.0, "C": 0.0, "D": 0.0}
assert abs(bias - 0.75) < 1e-9                        # 全押 B：最严重的位置偏差

rates, bias = position_bias_ex(["A", "B", "C", "D"])
assert abs(bias - 0.0) < 1e-9                         # 完全均匀：无偏差

rates, bias = position_bias_ex(["A", "A", "B", None, None])
assert abs(rates["A"] - 2 / 3) < 1e-9                 # None 不进分母
assert abs(bias - (2 / 3 - 0.25)) < 1e-9

assert position_bias_ex([None, None]) == ({"A": 0.0, "B": 0.0, "C": 0.0, "D": 0.0}, 0.0)
print("✅ 练习 3 通过")

## ✏️ 练习 4：实现 judge 打分文本的鲁棒解析 `parse_score_ex`

LLM judge 被要求输出 `Score: <0-5>`，但真实回复五花八门。实现 `parse_score_ex(text)`，返回 0–5 的 int 或 `None`，降级链：

1. **格式命中**：`Score: 4` / `score:3` / `score = 5`（关键词不区分大小写）——优先级最高，reasoning 里出现别的数字也不受干扰。
2. **分数形式**：`4/5` → 取分子。
3. **全文唯一孤立数字**：整段只出现一个孤立的 0–5 才返回；出现两个不同数字 → 含糊 → `None`。
4. 全部失败（含超范围如 `Score: 9`、`text=None`）→ `None`。

**提示**：①用 `re.search(r"score\s*[:：=]?\s*([0-5])\b", text, flags=re.IGNORECASE)`；③孤立数字用 `(?<!\d)([0-5])(?!\d)`。注意②必须在③之前——否则 `"4/5"` 会因出现 4 和 5 两个数字被③误判成含糊。约 12 行。

In [ ]:
import re

def parse_score_ex(text):
    # TODO: ① re.search 关键词 score 后跟 0-5（大小写不敏感，允许 : ： = 分隔）
    # TODO: ② "n/5" 形式取分子
    # TODO: ③ 全文唯一孤立 0-5 数字才返回；多个不同数字 → None
    # TODO: ④ 其余（含 None 输入、超范围）返回 None
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert parse_score_ex("Reasoning: covers all key elements.\nScore: 5") == 5
assert parse_score_ex("score:3") == 3
assert parse_score_ex("The answer mentions 2 cats correctly. Score: 4") == 4   # ①优先，不被 "2" 干扰
assert parse_score_ex("I would rate this 4/5.") == 4
assert parse_score_ex("I give it a 3.") == 3
assert parse_score_ex("It misses 1 key element, maybe 2 are wrong.") is None   # 两个数字 → 含糊
assert parse_score_ex("Score: 9") is None                                      # 超范围
assert parse_score_ex(None) is None
print("✅ 练习 4 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
import re

def extract_choice_ex(text, options=None):
    if text is None:
        return None
    t = text.strip()
    # ① 开头即字母（字母后不能紧跟其他字母，防 "Because" → B）
    m = re.match(r"^[\(\[]?([A-D])(?![A-Za-z])", t)
    if m:
        return m.group(1)
    # ② 模式词
    m = re.search(r"(?:answer|option|choice|答案|选项|选)\s*(?:is|：|:)?\s*[\(\[]?([A-D])\b",
                  t, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()
    # ③ 全文唯一孤立字母
    isolated = re.findall(r"(?<![A-Za-z])([A-D])(?![A-Za-z])", t)
    if len(set(isolated)) == 1:
        return isolated[0]
    # ④ 选项文本匹配
    if options:
        hits = ["ABCD"[i] for i, opt in enumerate(options)
                if opt and opt.lower() in t.lower()]
        if len(set(hits)) == 1:
            return hits[0]
    return None

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def cyclic_shift_ex(options, gold_letter, j):
    k = len(options)
    shifted = [options[(i - j) % k] for i in range(k)]
    new_gold_idx = ("ABCD".index(gold_letter) + j) % k
    return shifted, "ABCD"[new_gold_idx]

def run_circular_ex(items, answer_fn):
    n_correct = 0
    for it in items:
        ok = True
        for j in range(len(it["options"])):
            opts_j, gold_j = cyclic_shift_ex(it["options"], it["answer"], j)
            ok = ok and (answer_fn(it, opts_j) == gold_j)
        n_correct += ok
    return n_correct / len(items)

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def position_bias_ex(preds):
    valid = [p for p in preds if p is not None]
    if not valid:
        return {ch: 0.0 for ch in "ABCD"}, 0.0
    rates = {ch: valid.count(ch) / len(valid) for ch in "ABCD"}
    return rates, max(rates.values()) - 0.25

In [ ]:
# 练习 4 参考答案（先自己做，再对照）
import re

def parse_score_ex(text):
    if text is None:
        return None
    m = re.search(r"score\s*[:：=]?\s*([0-5])\b", text, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    m = re.search(r"(?<!\d)([0-5])\s*/\s*5\b", text)
    if m:
        return int(m.group(1))
    digits = re.findall(r"(?<!\d)([0-5])(?!\d)", text)
    if len(set(digits)) == 1:
        return int(digits[0])
    return None

## 小结 + 动手练习

**你刚刚搭出了一套方法论完整的 VLM 评测 harness：**
- **多选评测集**（手工 + 真实基准回退），统一格式，覆盖多维能力。
- **带兜底链的答案抽取**，把 `unparseable`（指令遵循失败）从真错里分离出来——这是评测最容易踩的坑。
- **CircularEval**：循环移位、全对才算对，量化并挤掉 **position bias**。
- **no-image 探针 + MG/ML 指标**：诊断语言先验 / 数据污染，判断分数是否"干净"。
- **LLM-as-Judge**：reasoning-before-score 的 rubric prompt，真 API 与本地占位双路径，并诚实声明占位的局限。

**练习（建议动手）：**
1. **加 bootstrap 置信区间**：对 `naive_records` 的逐题 `correct` 做 1000 次有放回重采样，输出 `naive_acc` 的 95% CI，写成 `xx.x% (CI a–b)` 的形式（讲解 §7）。
2. **prompt 鲁棒性**：把 `INSTRUCTION` 换 3 种同义改写各跑一遍，比较 accuracy 的方差——体会"单点分数不可信"。
3. **judge 一致性验证**：给 3 条 caption 手工标 0–5 分当人类标注，跑 `judge` 后算与人类的 Spearman 相关 / agreement，判断本地占位 judge 是否够格下结论（讲解 §5.3）。
4. **能力归因**：对一道答错的题，把图换成裁剪到关键区域的高清版重测；若答对则瓶颈在视觉编码器（分辨率），否则在 LLM 推理（讲解 §8）。

➡️ 下一站：**模块 08 — 幻觉、鲁棒性与安全评估**，把"答错"细分为"看不见 / 看错 / 编造（hallucination）/ 被攻破（jailbreak）"，并学 POPE、HallusionBench 等专项评测。

---
## 🎯 真实数据胶囊题：真实特征上的图文检索 Recall@K

VLM 检索评测的核心指标是 Recall@K：给一张图，正确文本是否在 top-K 最相似里。用真实图像区域特征构造图文对，实现 Recall@K，验证 K 越大召回越高。

> 本模块新增的**真实数据**练习：用**真实图像**把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, io, urllib.request
import numpy as np
import matplotlib.image as mpimg
CACHE=os.path.expanduser("~/.vlm_data"); os.makedirs(CACHE,exist_ok=True)
def real_image():
    "真实图像 Grace Hopper (来自 matplotlib 示例数据), 返回 (H,W,3) uint8"
    p=os.path.join(CACHE,"grace_hopper.jpg")
    if not os.path.exists(p):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/matplotlib/matplotlib/main/lib/matplotlib/mpl-data/sample_data/grace_hopper.jpg", p)
    return mpimg.imread(p)

img=real_image().astype(float)/255
rng=np.random.default_rng(0); B=20; d=48
I=np.stack([img[i*25:i*25+25].reshape(-1)[:d] for i in range(B)])
T=I + rng.normal(0,0.05,I.shape)    # 匹配文本=对应图像特征+噪声
I/=np.linalg.norm(I,axis=1,keepdims=True)+1e-9; T/=np.linalg.norm(T,axis=1,keepdims=True)+1e-9
print("图文检索 batch:", I.shape)

**练习**：实现 `recall_at_k(I, T, k)`：对每张图，看其匹配文本(同 index)是否在相似度 top-k 里，返回比例。

In [ ]:
def recall_at_k(I, T, k=1):
    # TODO: S=I@T.T; 对每行取 top-k 索引；命中 i==自己的比例
    raise NotImplementedError


In [ ]:
# 自测
r1=recall_at_k(I,T,1); r5=recall_at_k(I,T,5)
assert 0<=r1<=r5<=1, "Recall@K 随 K 单调不减"
assert r5>=r1
# 完美匹配 R@1=1
assert recall_at_k(I, I, 1)==1.0
print(f"图文检索 ✓  Recall@1={r1:.2f}  Recall@5={r5:.2f}")


### 📖 参考答案

In [ ]:
def recall_at_k(I, T, k=1):
    S=I@T.T; n=len(I); hit=0
    for i in range(n):
        topk=np.argsort(-S[i])[:k]
        hit += i in topk
    return hit/n
print("✓ Recall@K 是图文检索/CLIP 评测的标准指标")